In [26]:
# %%
import pandas as pd
import plotly.express as px

df = pd.read_excel('extract_constructions.xlsx', sheet_name='category_summary')
df

,verb,level,category,n,total,pct
0,like,A1,noun,631,1183,53.3
1,like,A1,none,263,1183,22.2
2,like,A1,pronoun,144,1183,12.2
3,like,A1,gerund,51,1183,4.3
4,like,A1,infinitive,44,1183,3.7
...,...,...,...,...,...,...
74,take,B1+,passive_subject,10,887,1.1
75,take,B1+,clausal,3,887,0.3
76,take,B1+,infinitive,3,887,0.3
77,take,B1+,gerund,1,887,0.1


In [27]:
# %%
level_order = ['A1', 'A2', 'B1+']
category_order = ['noun', 'proper_noun', 'coordinated_noun', 'pronoun',
                   'gerund', 'infinitive', 'clausal', 'predicate_adjective',
                   'passive_subject', 'phrasal_only', 'none']
verb_order = ['take', 'make', 'like']

orders = {'level': level_order, 'category': category_order, 'verb': verb_order}

# %%
plot_df = df.copy()
plot_df['n_label'] = 'n=' + plot_df['n'].astype(str)

def plot_verb_panel(df, verb, exclude_none=False):
    sub = df[df['verb'] == verb]
    if exclude_none:
        sub = sub[sub['category'] != 'none']
    fig = px.line(
        sub, x='level', y='pct', facet_col='category', facet_col_wrap=5,
        category_orders=orders, markers=True, text='n_label',
        hover_data={'n': True, 'total': True},
        labels={'pct': '% of sentences', 'level': ''},
        title=f'{verb}: category share by CEFR level',
    )
    fig.update_traces(textposition='top center', textfont_size=9)
    fig.update_yaxes(matches=None, showticklabels=True, title='')
    fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
    fig.update_layout(showlegend=False, height=450)
    return fig

# %%
plot_verb_panel(plot_df, 'take').show()
plot_verb_panel(plot_df, 'make').show()
plot_verb_panel(plot_df, 'like').show()

# No real clear patterns

# Noun collocations, these are the most frequent category by far
depending on CEFR level   
take: 70–72%   
make: 42–46%  
like: 52–54%  


# "take"

In [28]:
raw = pd.read_excel('extract_constructions.xlsx', sheet_name='raw_all')

# %%
noun_df = pd.read_excel('extract_constructions.xlsx', sheet_name='noun_collocates')

def biggest_movers(noun_df, verb, top_n=15, min_pct=0.5):
    sub = noun_df[noun_df['verb'] == verb]
    wide = sub.pivot_table(index='noun', columns='level',
                            values='pct_of_sentences', fill_value=0)
    wide = wide.reindex(columns=['A1', 'A2', 'B1+'], fill_value=0)
    # ignore nouns too rare to be worth reading into either way
    wide = wide[wide.max(axis=1) >= min_pct]
    wide['delta_A1_A2'] = wide['A2'] - wide['A1']
    wide['delta_A2_B1'] = wide['B1+'] - wide['A2']
    wide['delta_A1_B1'] = wide['B1+'] - wide['A1']
    return wide.sort_values('delta_A1_B1', ascending=False)

# %%
movers = biggest_movers(noun_df, 'take')
print('Rising toward B1+ (top):')
print(movers.head(top_n=10) if False else movers.head(10))
print('\nReceding toward B1+ (bottom):')
print(movers.tail(10))

Rising toward B1+ (top):
level       A1   A2  B1+  delta_A1_A2  delta_A2_B1  delta_A1_B1
noun                                                           
part       2.5  5.4  6.4          2.9          1.0          3.9
time       0.7  3.3  3.3          2.6          0.0          2.6
place      0.0  1.7  1.7          1.7          0.0          1.7
things     0.4  2.2  2.0          1.8         -0.2          1.6
care       0.7  0.8  2.0          0.1          1.2          1.3
water      1.4  0.9  2.5         -0.5          1.6          1.1
foods      1.1  2.0  2.1          0.9          0.1          1.0
dog        0.7  1.4  1.5          0.7          0.1          0.8
coat       0.0  0.2  0.8          0.2          0.6          0.8
breakfast  1.4  0.5  2.1         -0.9          1.6          0.7

Receding toward B1+ (bottom):
level       A1   A2  B1+  delta_A1_A2  delta_A2_B1  delta_A1_B1
noun                                                           
bicycle    0.7  0.0  0.0         -0.7          0

In [29]:
# examples of take part A2
raw[(raw['verb']=='take') & (raw['level']=='A2') & (raw['collocate']=='part')]['sentence'].sample(10, random_state=42).to_list()

['After the festival, I took part in the.',
 'So we need more groups which take part in our school festival.',
 'But at last almost of our classmates took part in this idea.',
 "To say truth, I couldn't take part is this festival this year.",
 'I wonder why eneyerse took part in school festival.',
 'But I took part in it in last year and two years ago.',
 'I took part in the contest and I got the first prize.',
 "I'll have to take part in KWF the next year because I am a champion!",
 'I also took part in.',
 "This year, I didn't take part in my school festival."]

In [30]:
# examples of take time A2
raw[(raw['verb']=='take') & (raw['level']=='A2') & (raw['collocate']=='time')]['sentence'].sample(10, random_state=42).to_list()

['I also like rice and miso soup, but it takes a time to eat.',
 'I think that it takes time to cook rice and soup.',
 'When I had my parents buy it, it took long time to persuade them.',
 'Because it is not take much time to have bread.',
 'By the way, I want to eat rice better, but it takes rather more time.',
 'I like bread more than rice, for I have to take a long time to eat rice.',
 "so, our's breakfast is bread and, milk tea, Because, It takes not much time to make them.",
 'Becaus bread is takes time.',
 'People say it takes time to cook rice cooker I want to sleep.',
 'But, it takes long time, so I have to get up earlier.']

In [21]:
# examples of take place A2
raw[(raw['verb']=='take') & (raw['level']=='A2') & (raw['collocate']=='place')]['sentence'].sample(10, random_state=42).to_list()

['So there is a bankbook at taking easy place.',
 'I have gone to bed by 10:30 PM but I usually go to bed at 1:00AM, for Festival Komaba will take place on November 1st, 2nd, 3rd, and we must prepare it.',
 'I want to take place a next year too.',
 'Our school festival took place on 14th, 15th September.',
 'Our school festival took place on September.',
 'If the big earthquake took place, I would bring out my pet at first.',
 'So we took place.',
 'Our school festival which is named "[proper noun] Festival "took place at September 15th and 16th.',
 'Our school festival take place in 15 September.',
 'It is tradition which has been taken place for long time.']

In [31]:
# examples of take time B1+
raw[(raw['verb']=='take') & (raw['level']=='B1+') & (raw['collocate']=='time')]['sentence'].sample(2, random_state=42).to_list()

['Rice and miso soup take much time.',
 'But it takes more time to cook rice in The morning.']

In [32]:
# examples or take money
raw[(raw['verb']=='take') & (raw['level']=='A1') & (raw['collocate']=='money')]['sentence'].sample(10, random_state=42).to_list()


['I would take out money and bankbook first thing.',
 'I take money and food.',
 'I take money, .',
 'I will take out money.',
 'I will take out money and bank book first.',
 'He worked, and he take much money.',
 'I would take out my money.',
 'I will take money and album',
 "Don't take a many money, , I take a food and water and.",
 "But, toyears I couldn't take new years money."]

# % of collocations by topic

In [33]:
# %%
def topic_distribution(raw_df, verb):
    sub = raw_df[raw_df['verb'] == verb]
    counts = sub.groupby(['level', 'topic']).size().reset_index(name='n')
    totals = sub.groupby('level').size().reset_index(name='total')
    dist = counts.merge(totals, on='level')
    dist['pct'] = (dist['n'] / dist['total'] * 100).round(1)
    wide = dist.pivot_table(index='topic', columns='level', values='pct', fill_value=0)
    wide = wide.reindex(columns=['A1', 'A2', 'B1+'], fill_value=0)
    return wide.sort_values('A1', ascending=False)

# %%
topic_distribution(raw, 'take')

level,A1,A2,B1+
topic,,,
earthquake,67.1,63.3,59.0
urashima,11.2,6.0,7.4
festival,10.1,12.8,14.4
dream,4.7,3.7,5.7
otoshidama,3.6,4.6,2.3
breakfast,3.2,9.6,11.2


### Breakfast sentences

In [35]:
# %%
for lvl in ['A1', 'A2', 'B1+']:
    sub = raw[(raw['verb']=='take') & (raw['topic']=='breakfast') & (raw['level']==lvl) &
              raw['category'].isin(['noun', 'proper_noun'])]
    print(f'--- take / breakfast / {lvl} (n={len(sub)}) ---')
    for _, row in sub.iterrows():
        print(f"{row['collocate']:15s} {row['sentence']}")
    print()

--- take / breakfast / A1 (n=9) ---
days            Because I take that days.
time            "It's takes too much time.
minutes         It takes a few minutes.
breakfast       I do not take breakfast, because I get up late.
bread           I think it is foolish to take bread or rice always.
breakfast       Usually, I try to take a breakfast every morning.
breakfast       I always take a breakfast on weekend.
breakfast       It's my presiouse time that take a "Japanese "breakfast in the warm sun light.
times           Because Eating Japanese foods takes many times.

--- take / breakfast / A2 (n=73) ---
lot             In the morning, I can't take a lot of breakfast because I wake up so slow.
train           Then I take the train.
bath            I take a bath, wear my school's uniform, and my tooth in the thirty minutes.
time            But, it takes long time, so I have to get up earlier.
time            It takes long time to eat.
time            If, breakfast is rice, I take much tim

### % of "take time" by topic and CEFR level

In [36]:
# %%
time_sub = raw[(raw['verb']=='take') & (raw['collocate']=='time')]
time_sub.groupby('level')['topic'].value_counts(normalize=True).round(3) * 100

level  topic     
A1     breakfast     50.0
       earthquake    50.0
A2     breakfast     93.5
       earthquake     6.5
B1+    breakfast     82.8
       festival      13.8
       earthquake     3.4
Name: proportion, dtype: float64

# "Make"

In [10]:
movers = biggest_movers(noun_df, 'make')
print('Rising toward B1+ (top):')
print(movers.head(top_n=10) if False else movers.head(10))
print('\nReceding toward B1+ (bottom):')
print(movers.tail(10))

Rising toward B1+ (top):
level   A1   A2  B1+  delta_A1_A2  delta_A2_B1  delta_A1_B1
noun                                                       
house  0.6  3.2  2.6          2.6         -0.6          2.0
movie  1.8  5.8  3.7          4.0         -2.1          1.9
gate   0.0  0.5  0.8          0.5          0.3          0.8
thing  0.0  0.6  0.8          0.6          0.2          0.8
art    0.0  0.3  0.8          0.3          0.5          0.8
lunch  0.3  0.5  1.0          0.2          0.5          0.7
a.     0.3  0.4  0.9          0.1          0.5          0.6
cakes  0.0  0.2  0.6          0.2          0.4          0.6
goal   0.0  0.3  0.6          0.3          0.3          0.6
mind   0.0  0.5  0.6          0.5          0.1          0.6

Receding toward B1+ (bottom):
level         A1   A2  B1+  delta_A1_A2  delta_A2_B1  delta_A1_B1
noun                                                             
pool         0.6  0.0  0.0         -0.6          0.0         -0.6
udon         0.6  0.2  0.0

In [40]:
# examples of make house A2
raw[(raw['verb']=='make') & (raw['level']=='A2') & (raw['collocate']=='house')]['sentence'].sample(30, random_state=42).to_list()

['In our school festival, we made a ghost house.',
 'our class make ghost house.',
 'In the school festival, our class made a ghost house.',
 'This year, our class made a fantom-house.',
 'But we could complete making a Japanese haunted house by the time.',
 "In my friend's class, they have made house for long time.",
 'Our class cooprated to make a ghost house for about two weeks before the festival.',
 'I was willing to make our special "gost house ".',
 'Our class made a "PHOTO HOUSE ".',
 'Our class made a ghost house.',
 'Our class made a ghost house.',
 'We made a Ghost House in classroom.',
 'Our class made the ghost house.',
 'Because it was dicided that we have to done about art, but we made a ghost-house of Egypt.',
 "As I belong to vollyball club, I couldn't help to make the ghost house so often, but I helped to make the ghost as much as I could.",
 'That is why, we wanted to make ghost house or foods.',
 'Our class make a old house.',
 "This year, our class didn't allow to 

In [57]:
# # %%
# raw[(raw['verb']=='make') & (raw['level']=='A2') & (raw['collocate']=='house')]['topic'].value_counts()

# %%
raw[(raw['verb']=='make') & (raw['collocate']=='house')]['topic'].value_counts()

topic
festival      56
urashima       5
earthquake     2
Name: count, dtype: int64

In [42]:
# examples of make movie A2
raw[(raw['verb']=='make') & (raw['level']=='A2') & (raw['collocate']=='movie')]['sentence'].sample(10, random_state=42).to_list()

['We made a movie for the school festival.',
 'Our class made a movie this year.',
 'This year, we made a movie.',
 'Our class made a movie for a school festival.',
 'This year, our class made a movie.',
 'This year our class made movie.',
 'Our class made a movie named "The festival of death ".',
 'Our class made a movie.',
 'I made movie last and one more last year.',
 'Because it is not interesting for me to make a movie.']

In [56]:
# # %%
# raw[(raw['verb']=='make') & (raw['level']=='A2') & (raw['collocate']=='house')]['topic'].value_counts()

# %%
raw[(raw['verb']=='make') & (raw['collocate']=='house')]['topic'].value_counts()

topic
festival      56
urashima       5
earthquake     2
Name: count, dtype: int64

In [50]:
# examples of make rice A1
raw[(raw['verb']=='make') & (raw['level']=='A1') & (raw['collocate']=='rice')]['sentence'].sample(5, random_state=42).to_list()

['So he wants to make rice.',
 'He make rice and vegitaveles.',
 'He made "Great rice".',
 'I can make rice and.',
 'He make rice and vegitaveles.']

# "Make" clausal

In [51]:
# %%
df[(df['verb']=='make') & (df['category']=='clausal')][['level', 'n', 'total', 'pct']]

,level,n,total,pct
28,A1,56,327,17.1
36,A2,293,1128,26.0
46,B1+,226,930,24.3


In [53]:
# %%
for lvl in ['A1', 'A2', 'B1+']:
    sub = raw[(raw['verb']=='make') & (raw['category']=='clausal') & (raw['level']==lvl)]
    print(f'--- make / clausal / {lvl} (n={len(sub)}) ---')
    for s in sub['sentence'].sample(min(12, len(sub)), random_state=1):
        print(s)
    print()

--- make / clausal / A1 (n=56) ---
And I like American pop music, walk-man makes me happy.
He made lost.
And his son grew up to go to kill He waked and he make His name is Momotaro.
Milk makes me strong.
Making them moved by ourselves was the happiest thing: Also, everyone in my class, this was a big too.
Although our class made funny sceans sometime, probabory, it was serious.
There are many programs, I make, in the disks.
Many make the room bright! fin
If you go there, it makes you feel to want to go Okinawa.
I make it rule to read newspaper having breakfast.
I, it make me sad. . .
make is use making.

--- make / clausal / A2 (n=293) ---
Their smile and beautiful mind make me happy.
But, after all it makes my condision bad.
And we make effect to make good things.
But Rice and makes me warm.
It makes me hungry in the mornings, but I msut bear to eat a lunch at an early hour.
And milk makes me tall.
It made me happy.
Baking them well made me so happy.
Make the movie is very difficult.


# "like"

In [15]:
movers = biggest_movers(noun_df, 'like')
print('Rising toward B1+ (top):')
print(movers.head(top_n=10) if False else movers.head(10))
print('\nReceding toward B1+ (bottom):')
print(movers.tail(10))

Rising toward B1+ (top):
level        A1    A2   B1+  delta_A1_A2  delta_A2_B1  delta_A1_B1
noun                                                              
bread      10.7  12.5  13.1          1.8          0.6          2.4
festival    1.5   2.3   3.9          0.8          1.6          2.4
tea         0.7   0.8   1.3          0.1          0.5          0.6
foods       0.2   0.5   0.6          0.3          0.1          0.4
natto       0.0   0.5   0.2          0.5         -0.3          0.2
soup        0.5   1.9   0.7          1.4         -1.2          0.2
food        1.1   1.6   1.3          0.5         -0.3          0.2
breakfast   0.6   1.2   0.6          0.6         -0.6          0.0
coffee      0.2   0.5   0.1          0.3         -0.4         -0.1
song        0.4   0.7   0.2          0.3         -0.5         -0.2

Receding toward B1+ (bottom):
level         A1    A2   B1+  delta_A1_A2  delta_A2_B1  delta_A1_B1
noun                                                               
bask

# "like" + infinitive

In [54]:
# %%
df[(df['verb']=='like') & (df['category']=='infinitive')][['level', 'n', 'total', 'pct']]

,level,n,total,pct
4,A1,44,1183,3.7
11,A2,158,1982,8.0
20,B1+,108,991,10.9


In [16]:
# %%
for lvl in ['A1', 'A2', 'B1+']:
    sub = raw[(raw['verb']=='like') & (raw['category']=='infinitive') & (raw['level']==lvl)]
    print(f'--- like / infinitive / {lvl} (n={len(sub)}) ---')
    for s in sub['sentence'].sample(min(8, len(sub)), random_state=1):
        print(s)
    print()

--- like / infinitive / A1 (n=44) ---
I like to take pictures.
I would like to listen to music everytime and everywhere because I will do.
I like to eat buying.
I like to sleep very much.
I likes to use my computer.
I would like to buy clothes with next year's.
I don't like to do."
It said "I'd like to invite Ryuguujo!"

--- like / infinitive / A2 (n=158) ---
Because I like to listen to music.
So, I'd like to bring money first.
I like to listen to music.
I like to read newspaper when I was having breakfast.
I like to eat bread on breakfast.
And I like to read books and comics.
At first, I didn't like to do my.
I like to play a sports game.

--- like / infinitive / B1+ (n=108) ---
And I like to read books very much.
I like to eat rice in the night and noon.
I like to eat.
Because I think I don't like to eat rice in the morning.
After all, I don't like to use a big money.
But I think foreigner may say that he or she likes to eat bread for breakfast.
These pictures are really my treasures

In [17]:
# %%
verbal_df = pd.read_excel('extract_constructions.xlsx', sheet_name='verbal_complements')

def biggest_movers_verbal(verbal_df, verb, category, top_n=15, min_pct=0.3):
    sub = verbal_df[(verbal_df['verb'] == verb) & (verbal_df['category'] == category)]
    wide = sub.pivot_table(index='complement', columns='level',
                            values='pct_of_sentences', fill_value=0)
    wide = wide.reindex(columns=['A1', 'A2', 'B1+'], fill_value=0)
    # ignore complements too rare to be worth reading into either way
    wide = wide[wide.max(axis=1) >= min_pct]
    wide['delta_A1_A2'] = wide['A2'] - wide['A1']
    wide['delta_A2_B1'] = wide['B1+'] - wide['A2']
    wide['delta_A1_B1'] = wide['B1+'] - wide['A1']
    return wide.sort_values('delta_A1_B1', ascending=False)

# %%
movers = biggest_movers_verbal(verbal_df, 'like', 'infinitive')
print('Rising toward B1+ (top):')
print(movers.head(10))
print('\nReceding toward B1+ (bottom):')
print(movers.tail(10))

Rising toward B1+ (top):
level        A1   A2  B1+  delta_A1_A2  delta_A2_B1  delta_A1_B1
complement                                                      
eat         0.6  2.6  3.3          2.0          0.7          2.7
listen      0.3  0.6  1.0          0.3          0.4          0.7
be          0.0  0.2  0.4          0.2          0.2          0.4
see         0.0  0.1  0.4          0.1          0.3          0.4
take        0.2  0.2  0.6          0.0          0.4          0.4
bring       0.0  0.1  0.3          0.1          0.2          0.3
carry       0.0  0.0  0.3          0.0          0.3          0.3
go          0.2  0.3  0.4          0.1          0.1          0.2
sing        0.3  0.0  0.5         -0.3          0.5          0.2
read        0.1  0.3  0.3          0.2          0.0          0.2

Receding toward B1+ (bottom):
level        A1   A2  B1+  delta_A1_A2  delta_A2_B1  delta_A1_B1
complement                                                      
be          0.0  0.2  0.4         

In [60]:
# %%
raw[(raw['verb']=='like') & (raw['category']=='infinitive') & (raw['collocate']=='eat')].groupby('level').size()

# %%
for lvl in ['A1', 'A2', 'B1+']:
    sub = raw[(raw['verb']=='like') & (raw['category']=='infinitive') & (raw['collocate']=='eat') & (raw['level']==lvl)]
    print(f'--- like to eat / {lvl} (n={len(sub)}) ---')
    for s in sub['sentence'].sample(min(52, len(sub)), random_state=1):
        print(s)
    print()

--- like to eat / A1 (n=7) ---
I like to eat rice in the breakfast.
But, I like to eat
I don't like to eat rice.
Because I don't like to eat bread in the morning.
But I don't eat too much and I don't like to eat meat, because I don't like very much.
I like to eat.
I like to eat buying.

--- like to eat / A2 (n=52) ---
I like to eat rice more than bread in breakfast.
Sometimes I don't like eat lunch
I like to eat them in the morning and have no complaint about it.
And I like to eat delicious food.
But, I don't like to eat at morning.
I don't like to eat bread in the morning.
I like to eat shokupan with ham.
I don't have Natoo, but I wouldn't like to eat it in the morning.
I like to eat them, too.
If there is a ham or with bread, I like to eat bread for breakfast.
I like to eat fried fish and Sashimi.
I like to eat bread in the morning.
because, I like very to eat.
But I don't like to eat rice in the morning.
I like to eat.
I'd like to eat rice rather than bread.
I like to eat but I can'

In [59]:
# %%
raw[(raw['verb']=='like')  & (raw['category']=='infinitive') & (raw['collocate']=='eat')]['topic'].value_counts()

topic
breakfast     87
earthquake     4
festival       1
Name: count, dtype: int64

# HD-D

In [78]:
# %%
from lexicalrichness import LexicalRichness

def hdd_table(raw_df, verb, collocate_categories=('noun', 'proper_noun')):
    sub = raw_df[(raw_df['verb'] == verb) &
                 raw_df['category'].isin(collocate_categories) &
                 raw_df['collocate'].notna()]
    by_level = {lvl: sub[sub['level'] == lvl]['collocate'].tolist()
                for lvl in ['A1', 'A2', 'B1+']}
    draws = min(len(v) for v in by_level.values())
    rows = []
    for lvl in ['A1', 'A2', 'B1+']:
        colls = by_level[lvl]
        lex = LexicalRichness(' '.join(colls))
        hdd_score = lex.hdd(draws=draws)
        rows.append({'level': lvl, 'n_tokens': len(colls),
                     f'HD-D (draws={draws})': round(hdd_score, 4),
                     'expected_types': round(hdd_score * draws, 2)})
    return pd.DataFrame(rows).set_index('level')

# %%
# hdd_table(raw, 'like') # including proper nouns
hdd_table(raw, 'like', collocate_categories=('noun',))       # nouns only


,n_tokens,HD-D (draws=521),expected_types
level,,,
A1,631,0.2599,135.43
A2,1073,0.2563,133.55
B1+,521,0.2894,150.80


In [73]:
# hdd_table(raw, 'take')
hdd_table(raw, 'take', collocate_categories=('noun',))       # nouns only


,n_tokens,HD-D (draws=198),expected_types
level,,,
A1,198,0.4596,91.00
A2,652,0.4544,89.98
B1+,640,0.4908,97.17


In [76]:
# hdd_table(raw, 'make')
hdd_table(raw, 'make', collocate_categories=('noun',))       # nouns only


,n_tokens,HD-D (draws=137),expected_types
level,,,
A1,137,0.6715,92.00
A2,502,0.6110,83.70
B1+,431,0.6440,88.22


# check "make" collocates

In [88]:
# %%
counts = raw[(raw['verb']=='make') & (raw['category']=='noun')]['collocate'].value_counts()
for noun, n in counts.items():
    print(f'{noun:15s} {n}')

movie           105
house           56
breakfast       43
friends         22
bread           22
festival        19
lot             18
story           15
lunch           14
rice            14
thing           14
a.              14
gate            13
foods           13
stage           12
mind            12
friend          11
food            11
money           10
art             10
film            9
time            9
memory          8
things          8
play            8
plan            8
goal            8
effort          8
cake            7
picture         6
game            6
mistake         6
memories        6
fire            6
mother          6
sound           5
events          5
book            5
programs        5
records         5
doughnut        5
break           5
bridge          5
cloth           5
stories         5
box             5
cakes           5
people          5
soup            4
group           4
fun             4
power           4
kind            4
movies          4
sandwic

In [91]:
# %%
make_nouns = raw[(raw['verb']=='make') & (raw['category']=='noun')]

for lvl in ['A1', 'A2', 'B1+']:
    print(f'--- make / noun / {lvl} ---')
    counts = make_nouns[make_nouns['level']==lvl]['collocate'].value_counts()
    for noun, n in counts.head(15).items():
        print(f'{noun:15s} {n}')
    print()

--- make / noun / A1 ---
movie           6
breakfast       6
festival        5
bread           5
rice            5
story           4
foods           4
time            3
lot             3
bridge          3
mother          3
mistake         2
memory          2
building        2
play            2

--- make / noun / A2 ---
movie           65
house           32
breakfast       22
friends         10
story           8
stage           8
lot             7
thing           7
bread           7
mind            6
festival        6
money           6
foods           6
rice            6
food            6

--- make / noun / B1+ ---
movie           34
house           22
breakfast       15
friends         10
bread           10
lunch           8
festival        8
lot             8
a.              8
thing           7
gate            7
art             7
mind            6
friend          5
doughnut        5

